In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. 定义路径
# 基础模型：你训练时用的那个（建议填 HF 上的 ID 或本地路径）
base_model_path = "Qwen/Qwen2.5-3B-Instruct" 
# 适配器路径：你本地的 ./saves 文件夹
# adapter_path = "/home/ha5083li/Downloads/llm_related/knowledge_distillation_llm/saves" 
adapter_path = os.path.abspath("/home/ha5083li/Downloads/llm_related/knowledge_distillation_llm/results_0321_exp14/checkpoint-300")
# 合并后保存的路径
save_to = "/home/ha5083li/Downloads/llm_related/knowledge_distillation_llm/results_0321_exp14/Qwen2.5-3B-ReasonLite-exp14"

print(f"正在加载基础模型: {base_model_path}")
# 注意：合并时建议使用与训练一致的精度（通常是 float16 或 bfloat16）
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.bfloat16, # 或者 torch.float16
    device_map="auto",
    trust_remote_code=True
)

# 2. 加载 LoRA 适配器
print(f"正在加载适配器: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. 执行合并并卸载
# 这个操作会将 LoRA 权重永久注入模型层，并删除 LoRA 特有的结构
print("正在合并权重...")
merged_model = model.merge_and_unload()

# 4. 保存完整模型和分词器
print(f"正在保存完整模型至: {save_to}")
merged_model.save_pretrained(save_to, safe_serialization=True)
tokenizer.save_pretrained(save_to)

print("✅ 合并成功！")